# Step 5 — LIME Explainability (Kaggle DistilBERT)

Retrains the Kaggle DistilBERT model (fast, ~6 min) in this session, then runs LIME on 10 REAL and 10 FAKE predictions to test whether the model relies on real semantic signal or dataset-specific artifacts.

**Research framing:** DistilBERT achieves near-perfect Kaggle accuracy. LIME is used to test whether this reflects genuine misinformation-detection signal or leakage from dataset-specific artifacts (source formatting, punctuation, structural patterns).

**Before running:** Runtime → Change runtime type → T4 GPU → Save. Upload `kaggle_clean.csv` to this session.

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas lime

## Imports and setup

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from lime.lime_text import LimeTextExplainer

print("GPU available:", torch.cuda.is_available())

MODEL_NAME = "distilbert-base-uncased"
LABEL_MAP = {"FAKE": 0, "REAL": 1}
LABEL_NAMES = ["FAKE", "REAL"]
MAX_LENGTH = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Retrain Kaggle DistilBERT (same setup as Step 4, subsampled to 15000 rows for speed)

In [ ]:
df = pd.read_csv("kaggle_clean.csv")
df = df.dropna(subset=["transformer_text", "binary_label"])
df["label"] = df["binary_label"].map(LABEL_MAP)
df = df[["transformer_text", "label", "binary_label"]].rename(columns={"transformer_text": "text"})
df = df.sample(n=15000, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True)).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True)).map(tokenize, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, pos_label=0),
        "recall": recall_score(labels, preds, pos_label=0),
        "f1": f1_score(labels, preds, pos_label=0),
    }

training_args = TrainingArguments(
    output_dir="./results_kaggle_distilbert_lime",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
print(f"\nTraining time: {(time.time()-start)/60:.1f} min")
print(trainer.evaluate())

model.to(device)
model.eval()

## LIME setup: prediction function

In [ ]:
explainer = LimeTextExplainer(class_names=LABEL_NAMES)  # index 0=FAKE, 1=REAL, matches LABEL_MAP

def predict_proba(texts):
    """LIME calls this repeatedly with perturbed versions of the input text."""
    inputs = tokenizer(
        list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs

## Select 10 REAL and 10 FAKE test examples, run LIME on each

In [ ]:
os.makedirs("lime_html_examples", exist_ok=True)

real_examples = test_df[test_df["binary_label"] == "REAL"].sample(n=10, random_state=1)
fake_examples = test_df[test_df["binary_label"] == "FAKE"].sample(n=10, random_state=1)
examples = pd.concat([real_examples, fake_examples]).reset_index(drop=True)

all_explanations = []

for i, row in examples.iterrows():
    text = row["text"][:2000]  # cap length -- LIME perturbs word-by-word, very long texts get slow
    true_label = row["binary_label"]

    exp = explainer.explain_instance(
        text, predict_proba, num_features=15, num_samples=300, labels=(0, 1)
    )

    pred_probs = predict_proba([text])[0]
    pred_label = LABEL_NAMES[np.argmax(pred_probs)]

    # Save individual HTML explanation
    html_path = f"lime_html_examples/example_{i}_{true_label}_pred_{pred_label}.html"
    exp.save_to_file(html_path)

    # Word weights for the predicted class
    for word, weight in exp.as_list(label=np.argmax(pred_probs)):
        all_explanations.append({
            "example_id": i, "true_label": true_label, "pred_label": pred_label,
            "confidence": float(max(pred_probs)), "word": word, "weight": weight,
        })

    print(f"[{i+1}/20] true={true_label} pred={pred_label} conf={max(pred_probs):.3f}")

lime_df = pd.DataFrame(all_explanations)
lime_df.to_csv("lime_explanations.csv", index=False)
print(f"\nSaved {len(lime_df)} word-weight rows to lime_explanations.csv")
print(f"Saved {len(examples)} HTML explanations to lime_html_examples/")

## Summary: top influential tokens per class

In [ ]:
fake_words = lime_df[lime_df["pred_label"] == "FAKE"].groupby("word")["weight"].agg(["mean", "count"]).sort_values("mean")
real_words = lime_df[lime_df["pred_label"] == "REAL"].groupby("word")["weight"].agg(["mean", "count"]).sort_values("mean", ascending=False)

print("=== Top 15 words pushing toward FAKE ===")
print(fake_words.head(15))

print("\n=== Top 15 words pushing toward REAL ===")
print(real_words.head(15))

summary_df = pd.concat([
    fake_words.head(15).reset_index().assign(direction="FAKE"),
    real_words.head(15).reset_index().assign(direction="REAL"),
])
summary_df.to_csv("lime_top_tokens_summary.csv", index=False)

## Artifact / leakage check

Flags whether top-weighted tokens look topical (semantic misinformation cues) or structural (formatting/source artifacts). This is the core check for your research question.

In [ ]:
# Heuristic artifact indicators: punctuation-like tokens, very short tokens,
# known structural words, all-digit tokens, single letters
ARTIFACT_PATTERNS = [
    r"^\d+$",           # pure numbers
    r"^[a-z]$",          # single letters
    r"^(said|says|according|photo|image|via|source)$",  # attribution/structural words
    r"^(mr|mrs|ms|dr)$", # titles (formatting, not content)
]

def is_likely_artifact(word):
    w = word.lower().strip()
    return any(re.match(p, w) for p in ARTIFACT_PATTERNS)

all_top_words = pd.concat([fake_words.reset_index(), real_words.reset_index()])
all_top_words["likely_artifact"] = all_top_words["word"].apply(is_likely_artifact)

artifact_count = all_top_words["likely_artifact"].sum()
total_count = len(all_top_words)

print(f"Top tokens flagged as likely structural/artifact: {artifact_count}/{total_count}")
print("\nFlagged tokens:")
print(all_top_words[all_top_words["likely_artifact"]][["word", "mean", "count"]])

print("\n--- Manual review needed ---")
print("This heuristic only catches obvious cases. Manually inspect lime_top_tokens_summary.csv")
print("and a few lime_html_examples/*.html files -- look for whether top words are topical")
print("(trump, election, government) vs structural (said, according, quote-marks, capitalization patterns).")

## Download everything

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("lime_results", "zip", ".", "lime_html_examples")
files.download("lime_explanations.csv")
files.download("lime_top_tokens_summary.csv")
files.download("lime_results.zip")